# Session 06 · Handling Different File Formats

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Setup

In [2]:
import io
from pathlib import Path
import numpy as np
import pandas as pd
OUT = Path("output"); OUT.mkdir(exist_ok=True)

## Example 1 - CSV with tricky options

In [3]:
text = "date;phone;city;amount\n05/10/2026;03001234567;Karachi;1,200\n06/10/2026;03219876543;Quetta;-\n"
df = pd.read_csv(io.StringIO(text), sep=";", dtype={"phone": str}, parse_dates=["date"],
                 dayfirst=True, thousands=",", na_values=["-"])
print(df, df.dtypes, sep="\n")
assert df.loc[0, "phone"].startswith("0") and df.loc[0, "date"].month == 10 and df["amount"].isna().sum() == 1

        date        phone     city  amount
0 2026-10-05  03001234567  Karachi  1200.0
1 2026-10-06  03219876543   Quetta     NaN
date      datetime64[us]
phone                str
city                 str
amount           float64
dtype: object


## Example 2 - Excel with two sheets

In [4]:
north = pd.DataFrame({"branch": ["Nazimabad", "Gulberg"], "sales": [120, 95]})
south = pd.DataFrame({"branch": ["Clifton", "Korangi"], "sales": [150, 80]})
with pd.ExcelWriter(OUT / "branches.xlsx") as xw:
    north.to_excel(xw, sheet_name="North", index=False)
    south.to_excel(xw, sheet_name="South", index=False)
sheets = pd.read_excel(OUT / "branches.xlsx", sheet_name=None)
print(list(sheets), pd.concat(sheets, names=["region"]).reset_index(level=0), sep="\n")

['North', 'South']
  region     branch  sales
0  North  Nazimabad    120
1  North    Gulberg     95
0  South    Clifton    150
1  South    Korangi     80


## Example 3 - nested JSON

In [5]:
data = [{"id": 1, "user": {"name": "Ali", "city": "Karachi"}, "tags": ["ai", "python"]},
        {"id": 2, "user": {"name": "Sana", "city": "Lahore"}, "tags": ["sql"]}]
flat = pd.json_normalize(data)
print(flat)
print(flat.explode("tags"))

   id          tags user.name user.city
0   1  [ai, python]       Ali   Karachi
1   2         [sql]      Sana    Lahore


   id    tags user.name user.city
0   1      ai       Ali   Karachi
0   1  python       Ali   Karachi
1   2     sql      Sana    Lahore


## Example 4 - XML

In [6]:
xml = """<stations>
  <station id="1"><name>Saddar</name><temp>34</temp></station>
  <station id="2"><name>Malir</name><temp>36</temp></station>
</stations>"""
st = pd.read_xml(io.StringIO(xml), xpath="//station")
print(st)

   id    name  temp
0   1  Saddar    34
1   2   Malir    36


## Example 5 - Parquet keeps types

In [7]:
sales = pd.DataFrame({"date": pd.date_range("2026-10-01", periods=4),
                      "city": pd.Categorical(["Karachi", "Lahore", "Karachi", "Multan"]),
                      "amount": [1200.0, 950.5, 1430.0, 800.0]})
sales.to_parquet(OUT / "sales.parquet")
sales.to_csv(OUT / "sales.csv", index=False)
print(pd.read_parquet(OUT / "sales.parquet").dtypes)
print(pd.read_csv(OUT / "sales.csv").dtypes)   # dates and categories become plain text

date      datetime64[us]
city            category
amount           float64
dtype: object
date          str
city          str
amount    float64
dtype: object


## Example 6 - a big file in chunks

In [8]:
rng = np.random.default_rng(6)
big = pd.DataFrame({"city": rng.choice(["Karachi", "Lahore", "Peshawar"], 20_000),
                    "amount": rng.integers(100, 1000, 20_000)})
big.to_csv(OUT / "big.csv", index=False)
total = 0
for chunk in pd.read_csv(OUT / "big.csv", chunksize=5_000):
    total += chunk.loc[chunk["city"] == "Karachi", "amount"].sum()
print("Karachi total:", total)
assert total == big.loc[big["city"] == "Karachi", "amount"].sum()

Karachi total: 3662168
